# TMDB Keyword Lexicon — Emotion & Sentiment Analysis

**Datasets used:**
| Dataset | Role |
|---|---|
| [TMDB Keyword Lexicon](https://www.kaggle.com/datasets/bdelanghe/tmdb-keyword-lexicon) | NRC emotion + VAD scores per keyword |
| [TMDB Movie VAD + Emotion Scores](https://www.kaggle.com/datasets/bdelanghe/tmdb-movie-vad-emotion-scores) | Movie-level data for filtering |

**Filter applied to all analyses:** non-adult movies with at least 10 votes (theatrical/streaming releases).  
Keyword frequencies are re-counted against this filtered pool — not the raw TMDB totals.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

EMOTIONS = ['anger','anticipation','disgust','fear','joy','sadness','surprise','trust']
VAD = ['vad_valence','vad_arousal','vad_dominance']

# Resolve input paths
def find(name):
    matches = sorted(Path('/kaggle/input').rglob(name))
    if not matches:
        raise FileNotFoundError(f'{name} not found under /kaggle/input')
    return matches[0]

kw_df  = pd.read_csv(find('tmdb_keyword_lexicon.csv'))
mov_df = pd.read_csv(find('movie_vad_scores.csv'), low_memory=False)

print(f'Keyword lexicon: {len(kw_df):,} keywords')
print(f'Movie scores:    {len(mov_df):,} movies')


## Filter to theatrical releases

In [ ]:
# Theatrical filter: non-adult, at least 10 votes
# adult column already excluded at build time in movie scores dataset,
# but we apply vote_count filter to focus on real releases
theatrical = mov_df[mov_df['vote_count'] >= 10].copy()
print(f'Theatrical pool: {len(theatrical):,} movies (vote_count >= 10)')
print(f'Excluded (< 10 votes): {len(mov_df) - len(theatrical):,}')

# Re-count keyword frequencies against theatrical pool only
kw_counter = Counter()
for cell in theatrical['keywords'].dropna():
    for kw in str(cell).split(', '):
        kw = kw.strip()
        if kw:
            kw_counter[kw] += 1

# Merge theatrical counts back into keyword lexicon
theatrical_counts = pd.DataFrame(kw_counter.items(), columns=['keyword','n_theatrical'])
kw = kw_df.merge(theatrical_counts, on='keyword', how='left')
kw['n_theatrical'] = kw['n_theatrical'].fillna(0).astype(int)

# Only keep keywords that appear in at least one theatrical release
kw_theatrical = kw[kw['n_theatrical'] > 0].copy()
print(f'\nKeywords in theatrical pool: {len(kw_theatrical):,} / {len(kw):,}')
print(f'Dropped (only in low-vote movies): {len(kw) - len(kw_theatrical):,}')


## 1. Schema & coverage

In [ ]:
print('Columns:', list(kw_theatrical.columns))
print(f'\nLexicon coverage (theatrical keywords only):')
for col in ['emolex_found','intensity_found','vad_found']:
    n = kw_theatrical[col].sum()
    print(f'  {col:20s}: {n:,} / {len(kw_theatrical):,} ({n/len(kw_theatrical)*100:.1f}%)')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
labels = ['EmoLex\n(binary)', 'Intensity\n(0–1)', 'VAD v2.1\n(bipolar)']
found_cols = ['emolex_found','intensity_found','vad_found']
for ax, col, label in zip(axes, found_cols, labels):
    vals = [kw_theatrical[col].sum(), (~kw_theatrical[col]).sum()]
    ax.pie(vals, labels=['Found','Not found'], autopct='%1.1f%%',
           colors=['#4CAF50','#EF5350'], startangle=90)
    ax.set_title(label)
plt.suptitle('Keyword coverage by lexicon (theatrical releases)', fontsize=13)
plt.tight_layout()
plt.show()


## 2. Most common keywords in theatrical releases

In [ ]:
top_kw = kw_theatrical.nlargest(30, 'n_theatrical')[['keyword','n_theatrical','vad_valence','vad_found']]

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#1a9850' if v > 0 else '#d73027' if v < 0 else '#888'
          for v in top_kw['vad_valence'].fillna(0)]
ax.barh(top_kw['keyword'][::-1], top_kw['n_theatrical'][::-1], color=colors[::-1])
ax.set_title('Top 30 keywords in theatrical releases\n(green = positive valence, red = negative)')
ax.set_xlabel('Number of movies')
plt.tight_layout()
plt.show()

print(top_kw.to_string(index=False))


## 3. Most positive & negative keywords

Filtered to keywords appearing in 50+ theatrical movies.

In [ ]:
popular_kw = kw_theatrical[
    (kw_theatrical['n_theatrical'] >= 50) &
    kw_theatrical['vad_found']
].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, label, ascending in [(axes[0], 'Most positive (valence)', False),
                              (axes[1], 'Most negative (valence)', True)]:
    top = popular_kw.nsmallest(20, 'vad_valence') if ascending else popular_kw.nlargest(20, 'vad_valence')
    color = '#d73027' if ascending else '#1a9850'
    ax.barh(top['keyword'][::-1], top['vad_valence'][::-1], color=color, alpha=0.8)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title(label)
    ax.set_xlabel('VAD valence (−1 to +1)')
    for i, (_, row) in enumerate(top[::-1].iterrows()):
        ax.text(row['vad_valence'] + (0.01 if row['vad_valence'] >= 0 else -0.01),
                i, f"  {row['n_theatrical']:,}", va='center', fontsize=7,
                ha='left' if row['vad_valence'] >= 0 else 'right')

plt.suptitle('Keywords with most extreme valence (50+ theatrical movies)', fontsize=13)
plt.tight_layout()
plt.show()


## 4. VAD space — keyword clusters

In [ ]:
vad_kw = kw_theatrical[
    kw_theatrical['vad_found'] &
    (kw_theatrical['n_theatrical'] >= 100)
].copy()

fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(vad_kw['vad_valence'], vad_kw['vad_arousal'],
                c=vad_kw['vad_dominance'], cmap='RdYlGn',
                s=np.sqrt(vad_kw['n_theatrical']) * 3,
                alpha=0.5, vmin=-1, vmax=1)
plt.colorbar(sc, ax=ax, label='dominance')
ax.axvline(0, color='gray', lw=0.7, linestyle='--')
ax.axhline(0, color='gray', lw=0.7, linestyle='--')
ax.set_xlabel('Valence (negative ← 0 → positive)')
ax.set_ylabel('Arousal (calm ← 0 → excited)')
ax.set_title('Keywords in VAD space\n(dot size = frequency, color = dominance)')

# Annotate extreme points
for _, row in vad_kw.nlargest(5, 'vad_valence').iterrows():
    ax.annotate(row['keyword'], (row['vad_valence'], row['vad_arousal']), fontsize=7, alpha=0.8)
for _, row in vad_kw.nsmallest(5, 'vad_valence').iterrows():
    ax.annotate(row['keyword'], (row['vad_valence'], row['vad_arousal']), fontsize=7, alpha=0.8)

plt.tight_layout()
plt.show()


## 5. Top keywords per emotion

Highest mean intensity score, filtered to 50+ theatrical movies.

In [ ]:
intensity_cols = [f'intensity_{e}' for e in EMOTIONS]
emo_kw = kw_theatrical[
    kw_theatrical['intensity_found'] &
    (kw_theatrical['n_theatrical'] >= 50)
].copy()

fig, axes = plt.subplots(2, 4, figsize=(16, 10))
for ax, emo in zip(axes.flat, EMOTIONS):
    col = f'intensity_{emo}'
    top = emo_kw.nlargest(12, col)[['keyword', col, 'n_theatrical']]
    ax.barh(top['keyword'][::-1], top[col][::-1], color='steelblue', alpha=0.8)
    ax.set_title(emo.capitalize())
    ax.set_xlabel('Intensity (0–1)')
    ax.tick_params(labelsize=8)

plt.suptitle('Top keywords per emotion (NRC Intensity, 50+ theatrical movies)', fontsize=13)
plt.tight_layout()
plt.show()


## 6. Emotion co-occurrence

Which emotions tend to appear together in the same keywords?

In [ ]:
emo_matrix = kw_theatrical[intensity_cols].dropna()
emo_matrix.columns = EMOTIONS
corr = emo_matrix.corr()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(EMOTIONS)))
ax.set_yticks(range(len(EMOTIONS)))
ax.set_xticklabels(EMOTIONS, rotation=45, ha='right')
ax.set_yticklabels(EMOTIONS)
for i in range(len(EMOTIONS)):
    for j in range(len(EMOTIONS)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax, label='Pearson r')
ax.set_title('Emotion intensity co-occurrence (keyword level)')
plt.tight_layout()
plt.show()


## 7. EmoLex binary distribution

How many keywords carry each binary emotion flag?

In [ ]:
emolex_cols = [f'emolex_{e}' for e in EMOTIONS]
emo_kw_binary = kw_theatrical[kw_theatrical['emolex_found']].copy()

counts = {e: emo_kw_binary[f'emolex_{e}'].sum() for e in EMOTIONS}
counts_s = pd.Series(counts).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(counts_s.index, counts_s.values, color='steelblue', alpha=0.8)
ax.set_title('Keywords with each EmoLex binary emotion flag\n(theatrical, emolex_found=True)')
ax.set_xlabel('Number of keywords')
for i, (emo, n) in enumerate(counts_s.items()):
    ax.text(n + 20, i, f'{n:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()


## Caveats

1. **Keyword frequencies** are re-counted against theatrical movies (vote_count ≥ 10) — not the raw TMDB totals in the lexicon's `n_movies` column.
2. **Scores are keyword associations**, not movie sentiment. "murder" scores high on anger because it associates with anger-related language, not because every movie tagged "murder" is angry.
3. **Bipolar composition caveat** — token-level averaging on a bipolar scale is lossy for phrases not in the NRC VAD MWE list. See: Mohammad et al., [SCL](https://www.saifmohammad.com/WebPages/SCL.html).
4. **~44% of theatrical keywords have no VAD coverage** and are excluded from VAD analyses.
